## Occupancy Data with VC analysis

We have a training set comprised of sensor data and a corresponding classification of whether or not a room is occupied. We will use the pocket algorithm to fit the data and analyze the generalization bounds using the VC dimension.

In [1]:
from __future__ import print_function
import pandas as pd
import numpy as np

import pocket as pk
import analysis_tools as at


# setup our training data
training = at.load_data(file_name='datatraining.txt', y_name='Occupancy')

# setup our testing data
testing = at.load_data(file_name='datatest.txt', y_name='Occupancy')

training.head()

,Temperature,Humidity,Light,CO2,HumidityRatio,Classification
1,23.18,27.2720,426.0,721.25,0.004793,1
2,23.15,27.2675,429.5,714.00,0.004783,1
3,23.15,27.2450,426.0,713.50,0.004779,1
4,23.15,27.2000,426.0,708.25,0.004772,1
5,23.10,27.2000,426.0,704.50,0.004757,1


## The VC_dimension

Before we actually train our classification alogrithm on our dataset, let's take a look at what the VC dimension can tell us about what we're about to do. 

---
**Theorem 2.5** (VC generalization bound).  For any $ \delta > 0,$

$$ E_{out}(g) \le E_{in}(g) + \sqrt{ \frac{8}{N} ln \frac{4mH(2N)}{\delta}} $$

With probability $\ge 1 - \delta $

Where 

$$ mH(2N)= (2 \times N)^{d_{vc}} +1 $$ 

is the growth function for perceptrons

---

What we are saying is that in order to know how well our algorithm will generalize to data that it has not seen($E_{out}$), we must frame it in terms of $E_{in}$ + (the result of our VC calcuation). This is a *generalization bound*, something *extremely* important in machine learning.

The important part is that our VC calculation is dependent on $N$(amount of data we train our algorithm with), and $\delta$(percentage error we are willing to tolerate). 

As an example, let us say that our $E_{in}$ is *.06* and the result of our VC calculation is *.04*, where our error tolerance $\delta$ was *.03*.

We would then be able to say

$$ E_{out} \le .06 + .04 \ with \ probability \ge 1 - .03 $$

This in effect says with 97% certainty, $E_{out}$ will be less than .10%. So we are 97% certain that our algorithm will missclassify no more than 10% of samples that it has not seen. 

Powerful stuff, but does it work? 


First let us make an important distinction, we will use $E_{test}$ as an estimation for $E_{out}$. $E_{test}$ is estimated using our best guess for 'w' on test samples, which the alogrithm did not see in training. In this way, $E_{test}$ is about what we might expect to see for $E_{out}$.  

Formally we will be using $E_{in}$ to put an upper bound on $E_{out}$, using Theorem 2.5. 


---
## Run 1

The maximum number of points that the perceptron algorithm can shatter is $d + 1$. This corresponds to the dimensionality of $X$, plus the free paramater that we add, effectively the making the $d_{vc}$ a measure of the number of parameters. 

So 

$$  \mathbb{X}^d = 5 $$
$$ \mathbb{X}^d + 1 = 6 $$
represents our scalar $x_0$, and dimensions of $x$:
$$ x_0; x_1,...,x_5 $$

Let's train the pocket algorithm with 100 samples and shoot for an error tolerance of 0.07. This will give us:

> **parameters**


| N  | $\delta$ | $d_{vc}$|
| ---| ----|--------|
| 100| 0.7 | 6      |




In [2]:
# pull 100 samples randomly from the training data
hundred_samples = training.sample(n=100)

# return the best weight vector 'w' from fitting the pocket algorithm to our data
fit = pk.pocket(hundred_samples, iterations=30)
w= fit['w']

# calculate the in sample error using the training data and our fitted 'w' obtained from the pocket algorithm
e_in = at.misclassified_count(w=w, data=training)

# same as above, only we test this weight vector on data the alogrithm didn't see, the training data
e_test = at.misclassified_count(w=w, data=testing)

# calculate the VC contribution to the generalization bound(see theorum 2.5 above)
# N = sample size
# tolerance = our error tolerance
# mH = the growth function, the maximum number of dichotomies our hypoethesis set can generate(linear perceptron)
vc_bound = at.vc_bound(N=100, tolerance=.07, d_vc=6)

# print results below
at.print_results(e_in=e_in, e_test=e_test, vc=vc_bound)



E_in number misclassified: 8143, percentage error: 1.0
E_test number misclassified: 2665, percentage error: 1.0
VC bound: 1.69317355513




As can be seen above, our algorithm failed miserably, and the VC bound isn't helping us much either. We would have essentially 2.69 percent chance that our algorithm will perform badly. Clearly, we will need to make adjustments

---
## Run 2

The natural thing to do at this point is to increase our N, this should improve our performance. 

> **parameters**


| N   | $\delta$ | $d_{vc}$|
| ----| ----|--------|
| 1000| 0.7 | 6      |


In [3]:
thousand_samples = training.sample(n=1000)

fit = pk.pocket(thousand_samples, iterations=30)
w=fit['w']

e_in = at.misclassified_count(w=w, data=training)
e_test = at.misclassified_count(w=w, data=testing)
vc_bound = at.vc_bound(N=1000, tolerance=.07, d_vc=6)

at.print_results(e_in=e_in, e_test=e_test, vc=vc_bound)



E_in number misclassified: 8143, percentage error: 1.0
E_test number misclassified: 2665, percentage error: 1.0
VC bound: 0.630244201276




Here we see that again our algorithm performed very badly in sample, but the VC bound is better, however that is of little help to us when our in sample error is so high. 

---
## A look at  results so far

We seem to be having some problems doing any kind of learning at all, what could be the issue? 

Let's take a look at our data set and see what the ratio is for the classes 

In [4]:
class_count  = training['Classification'].value_counts()
class_count

-1    6414
 1    1729
Name: Classification, dtype: int64

Hmm, so the ratio of classes is very uneven. So let's see what the ratio looks like in the case of our first run, where we just sampled 100 random points from the training data.  

In [5]:
hundred_samples = training.sample(n=100)
hundred_samples['Classification'].value_counts()

-1    79
 1    21
Name: Classification, dtype: int64

Looking at these counts, we can roughly see that the *occupied* class makes up only 23-26% of the samples. This will give the pocket algorithm problems as it will continually see 'outliers', making the fit troublesome.  

There are two ways to remedy the issue

1) Give it more and more data. At around 5000 samples for this data set we will get good convergence

**Golen rule of machine learning**

> data conquers all. 

**Silver rule**

> golden rule applies, assuming you have the computational horsepower to train on all that data

**corollary**
> often you don't, but you do the best you can:)

2) Choose your samples more carefully.  This is dangerous, and we must still randomly sample, but do so more intelligently. 

---
## Run 3

This time we will use 100 samples as in our first run, but this time we will choose 100 samples randomly, but this
time ensuring that we get 50 samples with classification *1*, and 50 with *-1*. We will see if this improves our in sample error. 

> **parameters** choosing an equal number of samples from each class


| N  | $\delta$ | $d_{vc}$|
| ---| ----|--------|
| 100| 0.7 | 6      |

In [6]:
# we now make sure to sample 50 points from each class
hundred_samples = pd.concat([training[training['Classification']==1].sample(n=50),
                      training[training['Classification']==-1].sample(n=50)],
                      axis=0).reset_index(drop=True)

fit = pk.pocket(hundred_samples, iterations=30)
w = fit['w']
e_in = at.misclassified_count(w=w, data=training)
e_test  = at.misclassified_count(w=w, data=testing)
vc_bound = at.vc_bound(N=100, tolerance=.07, d_vc=6)

at.print_results(e_in=e_in, e_test=e_test, vc=vc_bound)




E_in number misclassified: 600, percentage error: 0.0736829178435
E_test number misclassified: 205, percentage error: 0.0769230769231
VC bound: 1.69317355513




Well....not bad right?  We have achieved an error rate of 0.073 with just 50 examples from each class. However the generalization bound is telling us we still need to bring down the VC contribution. How do we ensure this? We must use more data to get a better generalization bound.  

---
## Run 4

We can now try appoach 2 where we increase N. We might as well give it all the samples and then examine the classification error and generalization bound. 

> **parameters**


| N   | $\delta$ | $d_{vc}$|
| ----| ----|--------|
| 8143| 0.7 | 6      |

In [7]:

fit = pk.pocket(training, iterations=30)
w = fit['w']

e_in = at.misclassified_count(w=w, data=training)
e_test = at.misclassified_count(w=w, data=testing)
vc_bound = at.vc_bound(N=len(training), tolerance=.07, d_vc=6)

at.print_results(e_in=e_in, e_test=e_test, vc=vc_bound)




E_in number misclassified: 558, percentage error: 0.0685251135945
E_test number misclassified: 184, percentage error: 0.06904315197
VC bound: 0.247267113266




## Conclusions 

If we formulate the equation of the generalization bound(Theorem 2.5) using our last run utilizing all of the data we wind up with:

$$ E_{out}(g) \le 0.068 +  0.24 \ with \ probability \ge 1 - .07 $$

*Thus we can be **93%** certain that we will misclassify no more than ~.30% of data we have not seen.*

In reality we are likely to do better than this, given our $E_{test}$, but remember that the generalization bound is an **upper** bound on the maximum number of misclassifications we can be expected to make for $E_{out}$.  


We can see that the generalization bound becomes much tighter as we increase the size of N. With a small sample size, N=100, the generalization bound is essentially meaningless.  

With data that is skewed towards one class, that is it contains many more examples of a particular class, it becomes even more important to increase sample size, or choose a more representative sample. It is always important to sample randomly when doing so. 

It is encouraging that the pocket algorithm can achieve an in sample and out of sample error rate of roughly 7%. This is comparable to performance of roughly 5% error for very powerful methods like Random Forests used in the paper. We achieve learning with high confidence of generalizing well. 
